# ⚡ 따라쓰기 실습. 에너지 공공데이터 분석

**AI로 분석하는 신재생에너지 · Day 2 · 울산대학교**

한국에너지공단의 실제 공공데이터로 **전국 지역별 신재생에너지 현황**을 분석합니다.

### 📌 이 파일 사용법

1. 각 단계의 **회색 상자 안 코드**를 아래 빈 셀에 **직접 타이핑**합니다
2. `Shift + Enter` 로 실행합니다
3. **에러가 나면 마지막 줄부터 읽으세요.** 에러는 실패가 아니라 다음에 할 일을 알려주는 안내입니다

> 벅스 차트에서 쓴 순서 그대로 갑니다 — 불러오기 → 살펴보기 → 골라내기 → 집계

---
## 1단계. 데이터 파일 올리기

분석할 CSV 파일을 코랩으로 올립니다.

```python
from google.colab import files              # 코랩의 파일 업로드 도구

uploaded = files.upload()                   # 실행하면 [파일 선택] 버튼 등장

파일명 = list(uploaded.keys())[0]           # 올린 파일의 이름을 자동으로 받아옴

print(파일명)                               # 어떤 이름으로 저장됐는지 확인
```

### ⚠️ 파일명을 직접 치지 않는 이유

한글 파일명은 **화면에 똑같이 보여도 컴퓨터 안에서는 다른 글자**일 수 있습니다.

이름을 변수에 받아 두면 이 문제가 사라집니다. 실무에서도 이렇게 합니다.

> 코랩은 세션이 끊기면 올린 파일이 사라집니다. 파일을 못 찾는다는 에러가 나오면 이 셀부터 다시 실행하세요.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 2단계. 데이터 불러오기

올린 CSV 파일을 판다스로 읽어 들입니다.

```python
import pandas as pd                         # 표 데이터를 다루는 도구, 별명은 pd

df = pd.read_csv(파일명)                    # CSV 파일 읽기
```

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 🔍 에러가 났습니다

마지막 줄을 읽어보세요 → `UnicodeDecodeError`

**글자를 해석하지 못했다는 뜻입니다.** 글자를 저장하는 규칙(인코딩)이 여러 가지인데,

판다스가 기본으로 가정한 규칙과 이 파일의 규칙이 다릅니다.

국내 공공기관 자료는 대부분 **cp949** 라는 옛 한글 규칙을 씁니다. 규칙을 알려주고 다시 읽어봅니다.

```python
df = pd.read_csv(파일명, encoding='cp949')  # 한글 인코딩 지정
```

💬 벅스 차트를 **저장**할 때 `utf-8-sig` 를 썼죠. 이번엔 **읽을 때** 인코딩입니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 3단계. 데이터 살펴보기

분석 전에 **무엇이 들어 있는지** 먼저 봅니다. 벅스에서 한 순서와 같습니다.

```python
df.head()    # 앞 5줄 — 어떤 열이 있는지 눈으로 확인
```

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


```python
df.info()    # 행 수 · 열 이름 · 빈 값 · 자료형 요약
```

**확인할 것**
- `3654 entries` — 이 정도면 눈으로 다 볼 수 없습니다. 그래서 코드로 봅니다
- 빈 값(Null)은 거의 없습니다 — 정부 통계라서 비교적 깨끗합니다

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 4단계. 발전량 데이터 꺼내보기

이 데이터에서 우리가 비교할 값은 **발전량(MWh)** 입니다. 그 열만 꺼내 봅니다.

```python
df['발전량(MWh)']        # 발전량 열만 선택
```

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 🔍 또 에러입니다

마지막 줄 → `KeyError`. **그런 이름의 열이 없다**는 뜻입니다.

방금 `head()` 에서 분명히 본 이름인데 왜 없다고 할까요? 열 이름을 **있는 그대로** 확인해 봅니다.

```python
df.columns               # 열 이름을 정확하게 확인
```

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 원인과 해결

출력을 자세히 보면 → `' 발전량(MWh) '` — **이름 앞뒤에 공백이 있습니다.**

사람 눈에는 안 보이지만 컴퓨터에게 `'발전량'` 과 `' 발전량 '` 은 완전히 다른 이름입니다.

모든 열 이름의 앞뒤 공백을 한 번에 정리합니다.

```python
df.columns = df.columns.str.strip()         # 모든 열 이름의 앞뒤 공백 제거

df['발전량(MWh)']                            # 이제 정상
```

💬 `strip` = 양끝을 벗겨낸다는 뜻입니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 5단계. 광역시도별 발전량 합계 구하기

지역마다 얼마나 만드는지 비교하려면 **광역시도 단위로 묶어서 합계**를 내야 합니다.

벅스에서 조건에 맞는 행을 골라냈다면, 이번에는 **묶어서 계산**하는 `groupby` 를 씁니다.

```python
df.groupby('광역')['발전량(MWh)'].sum().sort_values()   # 광역별 합계를 작은 순으로
```

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 🔍 결과를 확인해 봅니다

울산이 약 **4,503,000 MWh** 로 나옵니다.

그런데 어제 확인한 울산의 발전량은 **75만 MWh** 수준이었습니다. **약 6배 차이입니다.**

코드에는 문제가 없고 컴퓨터도 계산을 틀리지 않습니다.

**그렇다면 무엇이 잘못됐을까요?**

---
## 6단계. 숫자가 이상한 이유 찾기

합계가 커졌다는 것은 **같은 값을 여러 번 더했다**는 뜻입니다.

어떤 행들이 들어 있는지 확인해 봅니다. 먼저 에너지원 항목부터 봅니다.

```python
df['에너지원'].unique()      # 에너지원 열에 어떤 값들이 있는지 (중복 제거)
```

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 발견 ① — 합계 항목이 섞여 있습니다

태양광·풍력 같은 **개별 에너지원** 사이에
`'신·재생에너지'`, `'재생에너지'`, `'신에너지'` 라는 **합계 항목**이 함께 들어 있습니다.

(Day 1에서 배운 법령상 분류 — 재생 7종 + 신 3종 = 신재생 — 가 그대로 데이터에 있습니다)

다음은 지역 쪽을 봅니다. 부산 한 곳만 뽑아서 확인해 봅니다.

```python
df[(df['광역'] == '부산') & (df['기초'] == '부산')]   # 광역과 기초가 모두 '부산'인 행
```

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 발견 ② — 지역 합계 행이 따로 있습니다

`부산-부산` 행이 **14개** 나옵니다. 에너지원마다 하나씩 있는 **부산 전체의 합계 행**입니다.

그리고 이와 별도로 `부산-해운대구`, `부산-사하구` 같은 **구·군별 행**도 데이터에 들어 있습니다.

### 정리 — 이 데이터는 두 축이 겹쳐 있습니다

| 지역 축 | 에너지원 축 | 예시 |
|---|---|---|
| 시도 합계 | 합계 항목 | 부산 · 부산 · 신·재생에너지 |
| 시도 합계 | 개별 에너지원 | 부산 · 부산 · 태양광 |
| 구 · 군 | 개별 에너지원 | 부산 · 해운대구 · 태양광 |

5단계에서는 이 **세 종류를 전부 더했습니다.** 그래서 같은 발전량이 여러 번 계산된 것입니다.

> 📌 **데이터 구조를 모르고 집계하면, 틀린 답이 그럴듯하게 나옵니다.**

---
## 7단계. 합계 행만 골라 다시 집계하기

중복을 없애려면 **세 종류 중 하나만** 남기면 됩니다.

여기서는 **시도 합계 행 × 개별 에너지원** 을 남기기로 합니다.

```python
시도별 = df[df['광역'] == df['기초']]                                  # ① 시도 합계 행만 남기기

세부 = 시도별[~시도별['에너지원'].isin(['신·재생에너지', '재생에너지', '신에너지'])]   # ② 합계 항목 제외

지역별 = 세부.groupby('광역')['발전량(MWh)'].sum().sort_values()        # ③ 다시 집계

print(len(지역별))                           # 몇 개 지역이 나왔는지

지역별
```

💬 `~` 는 **조건의 반대**, `isin([...])` 은 **목록 중 하나와 일치**. 둘을 합치면 "목록에 없는 것만"이 됩니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 🔍 울산은 750,479 — 어제 값과 맞습니다

그런데 출력된 지역 수가 **16개** 입니다.

대한민국 광역시도는 **17개** 입니다. **어느 지역이 안 보입니까?**

---
## 8단계. 빠진 지역 찾기

목록에 **제주**가 없습니다. 제주의 행들이 어떻게 적혀 있는지 확인해 봅니다.

```python
df[df['광역'] == '제주']['기초'].unique()    # 제주의 기초 항목 확인
```

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 원인과 해결

출력 → `['제주도', '제주시', '서귀포시', '기타']`

제주의 합계 행은 기초가 `'제주'` 가 아니라 **`'제주도'`** 로 적혀 있습니다.

그래서 `광역 == 기초` 조건(제주 == 제주도?)에 걸리지 않아 통째로 빠졌습니다.

조건에 **또는**을 추가해 제주도 합계 행도 포함시킵니다.

```python
시도별 = df[(df['광역'] == df['기초']) | (df['기초'] == '제주도')]      # | 는 '또는'

세부 = 시도별[~시도별['에너지원'].isin(['신·재생에너지', '재생에너지', '신에너지'])]

지역별 = 세부.groupby('광역')['발전량(MWh)'].sum().sort_values(ascending=False)   # 큰 순으로

print(len(지역별))                           # 이제 17개

지역별                                       # 최종 순위
```

💬 `ascending=False` — 큰 값부터 나열. 순위표는 이렇게 봅니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 단위 바꿔 보기

숫자가 길어서 비교가 어렵습니다. **1,000 MWh = 1 GWh** 로 환산합니다.

```python
(지역별 / 1000).round(0)      # 전체 값에 한 번에 나눗셈 적용
```

💬 17개 값이 **한 번에** 바뀝니다. NumPy에서 본 "배열 전체가 한 번에 계산"이 그대로 작동합니다.

### 🤔 결과를 읽어 보기

1. 울산은 작은 쪽에서 몇 번째입니까?
2. 1위 지역과 울산의 차이는 몇 배쯤 됩니까?
3. 제주는 울산보다 큽니까, 작습니까?

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 9단계. 울산의 에너지원별 발전량 구하기

울산이 **무엇으로** 전기를 만드는지 봅니다.

벅스에서 좋아하는 가수를 골라냈던 그 방법 그대로, 조건만 바꿉니다.

```python
울산 = 세부[세부['광역'] == '울산']                                    # 울산 행만 골라내기

울산.sort_values('발전량(MWh)', ascending=False)[['에너지원', '발전량(MWh)']]   # 큰 순으로 정렬
```

### 🤔 결과를 읽어 보기

1. 울산의 1위 에너지원은 무엇입니까? 예상과 같았습니까?
2. 값이 **0인 에너지원**들도 보세요 — 없다는 것도 정보입니다
3. 이 구성을 보면 울산은 어떤 도시라고 말할 수 있습니까?

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 검산 — 계산이 맞았는지 확인하기

울산의 에너지원별 값을 전부 더하면 8단계의 울산 합계와 같아야 합니다.

```python
울산['발전량(MWh)'].sum()          # 직접 합산
```

**같은 숫자(750,479)가 나오면** 우리가 거쳐 온 계산 경로가 옳았다는 뜻입니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## ✅ 오늘 사용한 함수

| 함수 | 하는 일 |
|---|---|
| `pd.read_csv(파일, encoding='cp949')` | 한글 공공데이터 읽기 |
| `df.head()` · `df.info()` | 데이터 살펴보기 |
| `df.columns` · `.str.strip()` | 열 이름 확인 · 공백 정리 |
| `df['열'].unique()` | 어떤 값들이 있는지 |
| `df[조건]` · `~` · `\|` · `.isin([...])` | 행 골라내기 — 조건 · 반대 · 또는 · 목록 |
| `df.groupby('기준')['값'].sum()` | 묶어서 집계 |
| `sort_values(ascending=False)` | 큰 순으로 정렬 |

## ⚡ 오늘 만난 에러와 이상 신호

| 신호 | 원인 | 해결 |
|---|---|---|
| `UnicodeDecodeError` | 한글 인코딩 규칙 불일치 | `encoding='cp949'` |
| `KeyError` | 열 이름에 숨은 공백 | `df.columns.str.strip()` |
| 값이 6배로 커짐 | 합계 행과 개별 행의 중복 합산 | 필터로 한 종류만 남기기 |
| 지역이 16개만 나옴 | 제주 합계 행의 표기 불일치 | 조건에 `\|` 추가 |

**공통점** — 셋 다 코드의 잘못이 아니라 **데이터를 확인하지 않아서** 생긴 일입니다.

> 📌 데이터는 눈으로 확인하기 전까지 믿지 않는다.

### ⏭️ 다음 시간 — 시각화

오늘 만든 `지역별` 과 `울산` 을 **그대로 그래프로** 그립니다. 변수를 지우지 마세요.